In [26]:
from itertools import product
from dataclasses import dataclass
from typing import Dict, List, Tuple

@dataclass
class Environment:
    """An environment: reward table + coin probabilities."""
    name: str
    p_heads: float  # P(coin = H)
    # rewards[coin_outcome][action] -> reward
    # coin_outcome: 'H' or 'T', action: 'H' or 'T'
    rewards: Dict[str, Dict[str, float]]

    @property
    def p_tails(self) -> float:
        return 1.0 - self.p_heads

    def prob(self, obs: str) -> float:
        return self.p_heads if obs == 'H' else self.p_tails

    def reward(self, obs: str, action: str) -> float:
        return self.rewards[obs][action]

    def expected_reward(self, policy: Dict[str, str]) -> float:
        """E_e[U] for a deterministic policy {obs -> action}."""
        return sum(
            self.prob(obs) * self.reward(obs, policy[obs])
            for obs in ['H', 'T']
        )


In [27]:
def make_coin_toss_envs(p_heads_1: float = 0.5, p_heads_2: float = 0.5) -> Tuple[Environment, Environment]:
    """Create the Match and Reverse Tails environments."""
    env1 = Environment(
        name="Match",
        p_heads=p_heads_1,
        rewards={
            'H': {'H': 1.0, 'T': 0.0},
            'T': {'H': 0.0, 'T': 1.0},
        }
    )
    
    env2 = Environment(
        name="Reverse Tails",
        p_heads=p_heads_2,
        rewards={
            'H': {'H': 0.0, 'T': 1.0},
            'T': {'H': 0.5, 'T': 0.5},
        }
    )
    return env1, env2

def all_policies() -> List[Dict[str, str]]:
    """All 4 deterministic policies: {obs -> action}."""
    policies = []
    for a_h, a_t in product(['H', 'T'], repeat=2):
        policies.append({'H': a_h, 'T': a_t})
    return policies

def print_table(headers: List[str], rows: List[List], col_width: int = 14):
    """Print a formatted table."""
    header_str = "".join(h.center(col_width) for h in headers)
    print(header_str)
    print("─" * len(header_str))
    for row in rows:
        print("".join(
            (f"{v:.4f}" if isinstance(v, float) else str(v)).center(col_width)
            for v in row
        ))
    print()

def policy_name(policy: Dict[str, str]) -> str:
    return f"{policy['H']}{policy['T']}"

In [ ]:
@dataclass
class IBTuple:
    """A single (P_e, c_e, lambda_e) tuple in an infra-distribution."""
    env: Environment
    c: float = 0.0
    lam: float = 1.0  # lambda

    def contribution(self, utility: float) -> float:
        """lambda_e * (E_e[U] + c_e) for a given utility value."""
        return self.lam * (utility + self.c)

@dataclass
class InfraDistribution:
    """
    An infra-distribution H = {(P_e, c_e, lambda_e)} over environments.

    The infra-expectation is:
        Ẽ_H[U] = inf_e  lambda_e * (E_e[U] + c_e)
    """
    tuples: List[IBTuple]

    @property
    def envs(self) -> List[Environment]:
        return [t.env for t in self.tuples]

    def infra_ev(self, policy: Dict[str, str]) -> float:
        """Ẽ_H[U] = inf_e lambda_e * (E_e[U] + c_e)"""
        return min(
            t.contribution(t.env.expected_reward(policy))
            for t in self.tuples
        )
    
    def ev_glued(self, policy: Dict[str, str], obs: str, on_history_reward: float) -> float:
        """
        Ẽ_H[r ⊕_obs U] where r is a fixed on-history reward, and U is normal off-history.

        (r ⊕_obs U)(x) = r       if x is on the obs-branch
                        = U(x)   if x is on the ¬obs-branch

        So E_e[r ⊕_obs U] = P_e(obs)*r + P_e(¬obs)*reward(¬obs, policy[¬obs])
        Then take inf over tuples.
        """
        not_obs = 'T' if obs == 'H' else 'H'
        vals = []
        for t in self.tuples:
            ev = (t.env.prob(obs) * on_history_reward +
                  t.env.prob(not_obs) * t.env.reward(not_obs, policy[not_obs]))
            vals.append(t.contribution(ev))
        return min(vals)

    def infra_ev_0_glued(self, policy: Dict[str, str], obs: str) -> float:
        """Ẽ_H[0 ⊕_obs U] — zero reward on obs-branch, normal off-history."""
        return self.ev_glued(policy, obs, on_history_reward=0.0)

    def infra_ev_1_glued(self, policy: Dict[str, str], obs: str) -> float:
        """Ẽ_H[1 ⊕_obs U] — max reward on obs-branch, normal off-history."""
        return self.ev_glued(policy, obs, on_history_reward=1.0)

    def infra_prob(self, policy: Dict[str, str], obs: str) -> float:
        """P̃_H(obs) = Ẽ_H[1 ⊕_obs U] - Ẽ_H[0 ⊕_obs U]"""
        return (self.infra_ev_1_glued(policy, obs) -
                self.infra_ev_0_glued(policy, obs))

    def infra_ev_updated(self, policy: Dict[str, str], obs: str) -> float:
        """
        Ẽ_{H|obs}[U] = (Ẽ_H[U] - Ẽ_H[0 ⊕_obs U]) / (Ẽ_H[1 ⊕_obs U] - Ẽ_H[0 ⊕_obs U])

        The infra-expected value after updating on observation obs.
        """
        numerator = self.infra_ev(policy) - self.infra_ev_0_glued(policy, obs)
        denominator = self.infra_prob(policy, obs)
        if abs(denominator) < 1e-12:
            return 0.0
        return numerator / denominator

    def update_tuples(self, policy: Dict[str, str], obs: str) -> 'InfraDistribution':
        """
        Update each tuple on observation obs (eq. 10):

        (P_e, c_e, λ_e) →  (P_{e|obs},
                             (c_e + E_e[0⊕U] - (1/λ_e)Ẽ_H[0⊕U]) / P_e(obs),
                             λ_e * P_e(obs) / P̃_H(obs))
        """
        infra_0 = self.infra_ev_0_glued(policy, obs)
        infra_p = self.infra_prob(policy, obs)

        new_tuples = []
        for t in self.tuples:
            # Per-environment E_e[0 ⊕_obs U]
            ee_0_glued = self.ev_glued(policy, obs, 0.0)
            
            # c_new = (c_old + E_e[0⊕U] - (1/lam)*Ẽ_H[0⊕U]) / P_e(obs)
            p_e_obs = t.env.prob(obs)
            if abs(p_e_obs) < 1e-12 or abs(infra_p) < 1e-12:
                continue

            new_c = (t.c + ee_0_glued - (1.0 / t.lam) * infra_0) / p_e_obs
            new_lam = t.lam * p_e_obs / infra_p

            # Create updated environment conditioned on obs
            updated_env = Environment(
                name=f"{t.env.name}|{obs}",
                p_heads=1.0 if obs == 'H' else 0.0,
                rewards=t.env.rewards
            )
            new_tuples.append(IBTuple(env=updated_env, c=new_c, lam=new_lam))

        return InfraDistribution(tuples=new_tuples)

In [29]:
class InfraBayesianAgent:
    """
    An agent that uses infra-Bayesian decision theory to select policies
    under Knightian uncertainty.
    """

    def __init__(self, infra_dist: InfraDistribution):
        self.H = infra_dist
        self.policies = all_policies()

    def optimal_policy(self) -> Tuple[Dict[str, str], float]:
        """Find the policy maximizing infra-EV (maximin)."""
        best_pi = max(self.policies, key=lambda pi: self.H.infra_ev(pi))
        return best_pi, self.H.infra_ev(best_pi)

In [30]:
env1, env2 = make_coin_toss_envs()
policies = all_policies()

print("EXERCISE 0: Standard Bayesian Decision Theory (30-70 prior)")
p1, p2 = 0.3, 0.7
rows = []
for pi in policies:
    ev1 = env1.expected_reward(pi)
    ev2 = env2.expected_reward(pi)
    ev_prior = p1 * ev1 + p2 * ev2
    rows.append([policy_name(pi), ev1, ev2, ev_prior])

print_table(["Policy", "EV Env1", "EV Env2", "EV (30-70)"], rows)

best_bayesian = max(policies, key=lambda pi:p1 * env1.expected_reward(pi) + p2 * env2.expected_reward(pi))
best_ev = p1 * env1.expected_reward(best_bayesian) + p2 * env2.expected_reward(best_bayesian)
print(f"Optimal Bayesian policy: {policy_name(best_bayesian)}")
print(f"Optimal EV:              {best_ev:.4f}")


EXERCISE 0: Standard Bayesian Decision Theory (30-70 prior)
    Policy       EV Env1       EV Env2      EV (30-70)  
────────────────────────────────────────────────────────
      HH          0.5000        0.2500        0.3250    
      HT          1.0000        0.2500        0.4750    
      TH          0.0000        0.7500        0.5250    
      TT          0.5000        0.7500        0.6750    

Optimal Bayesian policy: TT
Optimal EV:              0.6750


In [31]:
print("EXERCISE 1: Knightian Uncertainty (Infra-EV = min over envs)")
H = InfraDistribution(tuples=[
    IBTuple(env=env1, c=0.0, lam=1.0),
    IBTuple(env=env2, c=0.0, lam=1.0),
])
agent = InfraBayesianAgent(H)

rows = []
for pi in policies:
    ev1 = env1.expected_reward(pi)
    ev2 = env2.expected_reward(pi)
    infra = H.infra_ev(pi)
    worst = "Env1" if ev1 <= ev2 else "Env2"
    rows.append([policy_name(pi), ev1, ev2, worst, infra])

print_table(["Policy", "EV Env1", "EV Env2", "Worst Env", "Infra-EV"], rows, col_width=13)

best_pi, best_infra = agent.optimal_policy()
print(f"  Optimal IB policy: {policy_name(best_pi)}")
print(f"  Optimal Infra-EV:  {best_infra:.4f}")

EXERCISE 1: Knightian Uncertainty (Infra-EV = min over envs)
    Policy      EV Env1      EV Env2     Worst Env     Infra-EV  
─────────────────────────────────────────────────────────────────
      HH         0.5000       0.2500        Env2        0.2500   
      HT         1.0000       0.2500        Env2        0.2500   
      TH         0.0000       0.7500        Env1        0.0000   
      TT         0.5000       0.7500        Env1        0.5000   

  Optimal IB policy: TT
  Optimal Infra-EV:  0.5000


In [32]:
print("EXERCISE 2: Infra-EV of Glued Utilities & Infra-Probability")

for obs in ['T', 'H']:
    not_obs = 'H' if obs == 'T' else 'T'
    print(f"\n  Observation = {obs} (off-history = {not_obs})")
    rows = []
    for pi in policies:
        infra = H.infra_ev(pi)
        e0 = H.infra_ev_0_glued(pi, obs)
        e1 = H.infra_ev_1_glued(pi, obs)
        ip = H.infra_prob(pi, obs)
        rows.append([policy_name(pi), infra, e0, e1, ip])

    print_table(
        ["Policy", "Ẽ_H[U]", f"Ẽ[0⊕{obs} U]", f"Ẽ[1⊕{obs} U]", f"P̃_H({obs})"],
        rows
    )

EXERCISE 2: Infra-EV of Glued Utilities & Infra-Probability

  Observation = T (off-history = H)
    Policy        Ẽ_H[U]       Ẽ[0⊕T U]      Ẽ[1⊕T U]      P̃_H(T)    
──────────────────────────────────────────────────────────────────────
      HH          0.2500        0.0000        0.5000        0.5000    
      HT          0.2500        0.0000        0.5000        0.5000    
      TH          0.0000        0.0000        0.5000        0.5000    
      TT          0.5000        0.0000        0.5000        0.5000    


  Observation = H (off-history = T)
    Policy        Ẽ_H[U]       Ẽ[0⊕H U]      Ẽ[1⊕H U]      P̃_H(H)    
──────────────────────────────────────────────────────────────────────
      HH          0.2500        0.0000        0.5000        0.5000    
      HT          0.2500        0.2500        0.7500        0.5000    
      TH          0.0000        0.0000        0.5000        0.5000    
      TT          0.5000        0.2500        0.7500        0.5000    



In [33]:
print("EXERCISE 3: Updated Infra-Expectations Ẽ_{H|obs}[U]")

rows = []
for pi in policies:
    ev_t = H.infra_ev_updated(pi, 'T')
    ev_h = H.infra_ev_updated(pi, 'H')
    rows.append([policy_name(pi), ev_t, ev_h])

print_table(["Policy", "Ẽ_{H|T}[U]", "Ẽ_{H|H}[U]"], rows)

EXERCISE 3: Updated Infra-Expectations Ẽ_{H|obs}[U]
    Policy      Ẽ_{H|T}[U]    Ẽ_{H|H}[U]  
──────────────────────────────────────────
      HH          0.5000        0.5000    
      HT          0.5000        0.0000    
      TH          0.0000        0.0000    
      TT          1.0000        0.5000    



In [ ]:
print("EXERCISE 4: Updated Tuples (c, λ) After Observation")

for obs in ['T', 'H']:
    print(f"\n  After observing {obs}:")
    rows = []
    for pi in policies:
        updated = H.update_tuples(pi, obs)
        c1, lam1 = updated.tuples[0].c, updated.tuples[0].lam
        c2, lam2 = updated.tuples[1].c, updated.tuples[1].lam
        rows.append([policy_name(pi), c1, lam1, c2, lam2])

    print_table(
        ["Policy", "c₁", "λ₁", "c₂", "λ₂"],
        rows
    )